# Primaries, secondaries and longitudinal pairs
Includes code to generate figure 5.

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', None)
import numpy as np
import scipy.stats
from pathlib import Path
import sys
sys.path.append('../../src')
import data_imports
import matplotlib as mpl
from seaborn_helper_functions import *

Path("out").mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_rows', None)
np.random.seed(47)
set_plot_defaults(fontsize=7)

In [ ]:
BIOSAMPLES = data_imports.import_biosamples()
# BIOSAMPLES.head()
BIOSAMPLES.tumor_history.unique()

# Count primary and secondary tumors with ecDNA (5a-b)
ecDNA:  
202 of 2448 (8.3%) primary tumors amplified.  
57 of 519 (11.0%) secondary tumors amplified.  

chromosomal:  
129 of 2448 (5.3%) primary tumors amplified.  
33 of 519 (6.4%) secondary tumors amplified.  

## Stratified permutation test
Is ecDNA more frequent in secondary tumors, conditional on tumor type?  

ecDNA: p=0.002  
chromosomal: p=0.445

In [ ]:
def setup_stratified_permutation_test_data(biosamples,amp='ecDNA'):
    df = biosamples.copy()
    # Aggregate secondary tumor annotations
    df["history"] = df.tumor_history.map({
        'Diagnosis':'primary',
        'Recurrence':'secondary',
        'Relapse':'secondary',
        'Progressive':'secondary',
        'Metastasis':'secondary',
        'Second Malignancy':pd.NA,
        'Autopsy':pd.NA
    })
    # drop second malignancies, autopsies, and unannotated tumors.
    df = df.dropna(subset='history')

    # Collapse to at most one primary and one secondary per tumor
    df[amp] = df.amplicon_class==amp
    df = (
        df.groupby(["patient_id", "history", "cancer_type"], as_index=False)
          .agg(amp=(amp, "max"))   # OR across samples
    )
    
    # Keep tumor types with at least one primary and one secondary
    df = df.groupby("cancer_type").filter(
        lambda x: x.history.nunique() == 2
    ).reset_index(drop=True)
    return df

def compute_statistic(df):
    # amp should be either 'ecDNA' or 'intrachromosomal'
    p_sec = df.loc[df.history == "secondary",'amp'].mean()
    p_pri = df.loc[df.history == "primary",'amp'].mean()
    return p_sec - p_pri

def permute(df):
    dfp = df.copy()
    # paired tumors: swap labels with 50% probability
    for patient_id,pid_idx in dfp.groupby("patient_id").groups.items():
        if len(pid_idx) == 2:
            if np.random.rand() < 0.5:
                dfp.loc[pid_idx,'history'] = (
                    dfp.loc[pid_idx, "history"].iloc[::-1].values
                )
    # unpaired tumors: shuffle within cancer type
    for cancer_type, idx in dfp.groupby("cancer_type").groups.items():
        idx = np.array(list(idx)) # index of samples of this cancer type
        dfp.loc[idx,'history'] = np.random.permutation(
            dfp.loc[idx,'history'].values
        )
    return dfp

def permutation_test(df,N=1000,verbose=False):
    T_perm = np.empty(N)
    for i in range(N):
        if verbose and i%10==0:
            print("Iteration:",i)
        T_perm[i] = compute_statistic(permute(df))
    T_obs = compute_statistic(df)
    p = np.mean(T_perm >= T_obs)
    return p

def summarize(df):
    a = len(df[df.amp & (df.history == 'primary')])
    b = len(df[(df.history == 'primary')])
    c = len(df[df.amp & (df.history == 'secondary')])
    d = len(df[(df.history == 'secondary')])
    print(f"{a} of {b} ({round(a/b*100,1)}%) primary tumors amplified.")
    print(f"{c} of {d} ({round(c/d*100,1)}%) secondary tumors amplified.")

In [ ]:
df = setup_stratified_permutation_test_data(BIOSAMPLES)
summarize(df)
#permutation_test(df)

In [ ]:
df2 = setup_stratified_permutation_test_data(BIOSAMPLES,amp='intrachromosomal')
summarize(df2)
#permutation_test(df2)

In [ ]:
def plot_primary_secondary_amp_status2(df,ylabel=None,color_map=None,group_low_freq_tumor_types=11):
    '''
    Barplot illustrating the above result, colored by tumor type.
        ylabel: should be 'ecDNA' or 'chromosomal'
        color_map: generated automatically if not provided
        group_low_freq_tumor_types: if nonzero, group rare types into 'other' class to simplify plot. n=non-other classes.
    TODO: badly in need of a refactor.
    '''
    counts = (
        df.groupby(["history", "cancer_type"])["amp"]
          .agg(n="size", amp_true="sum")
          .reset_index()
    )

    totals = counts.groupby("history")["n"].sum()
    counts["height"] = counts["amp_true"] / counts["history"].map(totals)
    pivot = counts.pivot(index="history", columns="cancer_type", values="height").fillna(0)
    if group_low_freq_tumor_types > 0:
        if color_map is None:
            order = pivot.sum(axis=0).sort_values(ascending=False).index.tolist()
            pivot['Other']=pivot[order[group_low_freq_tumor_types:]].sum(axis=1)
            pivot=pivot.loc[:,pivot.columns.isin(order[:group_low_freq_tumor_types]+['Other'])].copy()
        else:
            pivot['Other']=pivot.loc[:,~pivot.columns.isin(color_map.keys())].sum(axis=1)
            pivot=pivot.loc[:,pivot.columns.isin(color_map.keys())].copy()
            
    tumor_types = list(pivot.columns)
    pivot = pivot.loc[:, (pivot != 0).any(axis=0)]

    order = pivot.sum(axis=0).sort_values(ascending=False).index.tolist()
    if 'Other' in order:
        order = [x for x in order if x != 'Other'] + ['Other']
    pivot = pivot[order]
    tumor_types = list(pivot.columns)
    if color_map is None:
        if len(tumor_types) <= 20:
            colors = sns.color_palette("tab20")[:len(tumor_types)]
        else:
            colors = sns.color_palette("husl", len(tumor_types))
        color_map = dict(zip(tumor_types, colors))
    
    colors = [color_map[d] for d in pivot.columns]
    sf=0.7
    fig, ax = plt.subplots(figsize=(0.75 * len(pivot)*sf, 3*sf))

    bottom = np.zeros(len(pivot))
    
    for d, color in zip(pivot.columns, colors):
        ax.bar(
            pivot.index,
            pivot[d],
            bottom=bottom,
            width=0.9,
            color=color,
            label=str(d),
        )
        bottom += pivot[d].values
    
    ax.set_xlabel(None)
    ax.set_ylabel(f"Fraction with {ylabel}")
    ax.set_ylim(0, 0.15)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ncol=1 if group_low_freq_tumor_types > 0 and group_low_freq_tumor_types < 12 else 2
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False,ncol=ncol)
    mpl.rcParams["ytick.left"] = True
    
    return fig,ax,color_map


In [ ]:
fig,ax,color_map = plot_primary_secondary_amp_status2(df,"ecDNA")
plt.tight_layout()
plt.show()
savefig(fig,'out/ecDNA-frac-primary-secondary')

In [ ]:
fig,ax,color_map = plot_primary_secondary_amp_status2(df2,"chromosomal amp.",color_map)
plt.tight_layout()
plt.show()
savefig(fig,'out/chromosomal-frac-primary-secondary')

# Longitudinal tumors (5c-d)

In [ ]:
pd.set_option('display.max_rows', 5)
SECONDARIES = ['Diagnosis','Progressive','Autopsy','Recurrence','Relapse','Metastasis'] #exclude second malignancies, no sample, unavailable
def get_sj_pairs():
    '''
    Longitudinal case from SJ or PNOC defined as having a diagnosis sample and a non-diagnosis sample.
    '''
    df = BIOSAMPLES[(BIOSAMPLES.cohort.str.startswith("SJ") | (BIOSAMPLES.cohort == "PNOC")) &
                    (BIOSAMPLES.duplicated('patient_id',keep=False))]
    grp = df.groupby('patient_id').filter(lambda x: x['tumor_history'].nunique() >= 2).sort_values(["patient_id","tumor_history"])
    return grp
def get_cbtn_pairs():
    '''
    Longitudinal case from CBTN defined as having samples with different dates of diagnosis (more than 1 month).
    '''
    df = BIOSAMPLES[BIOSAMPLES.cohort.isin(["OpenPBTA","PBTA-X01"]) &
                    (BIOSAMPLES.tumor_history.isin(SECONDARIES)) &
                    (BIOSAMPLES.duplicated('patient_id',keep=False))]
    grp = df.groupby('patient_id').filter(lambda x: x['age_at_diagnosis'].max()-x['age_at_diagnosis'].min()>=30).sort_values(["patient_id","age_at_diagnosis"])
    return grp
def get_longitudinal_cases(verbose=True):
    df = pd.concat([get_cbtn_pairs(),get_sj_pairs()])
    if verbose:
        a = df.patient_id.nunique()
        b = df[df.amplicon_class == 'ecDNA'].patient_id.nunique()
        print(f"{b} of {a} longitudinal cases have ecDNA")
    return df

#get_sj_pairs()
#get_cbtn_pairs()
longitudinal_cases = get_longitudinal_cases()
longitudinal_cases
# 5/2024: 18 of 85 longitudinal cases have ecDNA
# 9/2024: 31 of 213 longitudinal cases have ecDNA
# 1/2026: 28 of 213

In [ ]:
longitudinal_cases[longitudinal_cases.amplicon_class=='ecDNA'].patient_id.unique()

In [ ]:
longitudinal_cases.to_excel("out/longitudinal_cases.xlsx")

In [ ]:
def get_suppl_tbl_7():
    return pd.read_excel(data_imports.SUPPLEMENTARY_TABLES_PATH,sheet_name="10. Paired biosamples")

In [ ]:
df = get_suppl_tbl_7()
df.groupby('evolution_class').count()

## Notes on tumors with multiple ecDNAs
- PT_7WYPEC3Q (SHH MBL, primary -> progressive)
  - (loss) chr17p11.2, CN 15 
  - (loss) chr17:28,683,354-29,500,780 (chr17q), CN 14
  - (gain) TERT, CN 26
  - (gain) PPM1D, CN 13
- PT_KTRJ8TFY (H3K27 DMG, primary -> progressive)
  - (gain) PICALM, CN 6
  - (gain) FLT3 2x / CDX2, CN 15
- PT_XA98HG1C (SHH MBL, primary -> progressive)
  - (gain) MYCN, CN 54
  - (loss) FHL2 partial, CN 7
  - (recombinant) CCND2 partial, CN 15 -> 118
- SJ004912 (OS, primary -> metastasis)
  - (loss) amp6 no oncogenes, CN 12
  - (loss) amp9 chr7 no oncogenes, only LOC124901577, CN 2
  - (loss) amp10, probable FP
- SJ000912 (RMS, primary -> relapse)
  - (loss) CDKN1B
  - (conserved) MDM2

# AmpliconSimilarity Inputs

In [ ]:
# TODO: these samples still need longitudinal analyses done.
def get_ampliconsimilarity_todos():
    df = get_suppl_tbl_7()
    a = set(df.patient_id)
    df = get_longitudinal_cases(False)
    b = set(df[df.amplicon_class=='ecDNA'].patient_id)
    return df[(~df.patient_id.isin(a)) & df.patient_id.isin(b)]
pd.set_option('display.max_rows', None)
get_ampliconsimilarity_todos()

In [ ]:
# Generate required input file for ampliconsimilarity: features_to_graph.txt file of the biosamples for longitudinal cases.

def get_features_to_graph(file='../../data/source/AmpliconClassifier/pedpancan_features_to_graph.txt'):
    df = pd.read_csv(file,sep='\t',header=None,names=['bed','graph'])
    return df
def subset_features_to_graph(pairs_df):
    #pairs_df = get_longitudinal_cases(verbose=False)
    ftg_df = get_features_to_graph()
    for pt in pairs_df.patient_id.unique():
        print(pt)
        bs_set = pairs_df[pairs_df.patient_id == pt].index
        ftg_subset = ftg_df[ftg_df.bed.str.contains('|'.join(bs_set))]
        if len(ftg_subset) > 1:
            print("cp2")
            filepath=f'out/{pt}_features_to_graph.txt'
            ftg_subset.to_csv(filepath,sep='\t',header=False,index=False)
        else:
            print("cp3")
            continue
    return
subset_features_to_graph(get_ampliconsimilarity_todos())

In [ ]:
# Parse output

def get_feature_similarity_scores(file="../../data/source/AmpliconClassifier/pedpancan_feature_similarity_scores.tsv"):
    df = pd.read_csv(file,sep='\t')
    return df
df = get_feature_similarity_scores()

In [ ]:
pd.set_option('display.max_rows', None)
df.head()

# AmpliconSimilarity outputs
(Requires running CycleViz on the sample(s) of interest)

In [ ]:
# Get AS outputs
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


class Breakpoint(object):
    def __init__(self, lchrom, lpos, lstrand, rchrom, rpos, rstrand, cn):
        self.lchrom = lchrom
        self.lpos = int(lpos)
        self.lstrand = lstrand
        self.rchrom = rchrom
        self.rpos = int(rpos)
        self.rstrand = rstrand
        self.cn = cn

    def __str__(self):
        return f"{self.lchrom}:{str(self.lpos)}{'+' if self.lstrand else '-'}|{self.rchrom}:{str(self.rpos)}{'+' if self.rstrand else '-'}"

    def d_similar(self, bp2, d):
        bp2_chrom_set = {bp2.lchrom, bp2.rchrom}
        if self.lchrom not in bp2_chrom_set or self.rchrom not in bp2_chrom_set:
            return False

        sbp1 = sorted([(self.lchrom, self.lpos, self.lstrand), (self.rchrom, self.rpos, self.rstrand)])
        sbp2 = sorted([(bp2.lchrom, bp2.lpos, bp2.lstrand), (bp2.rchrom, bp2.rpos, bp2.rstrand)])
        
        # if chromosomes and strands are the same
        if sbp1[0][0] == sbp2[0][0] and sbp1[1][0] == sbp2[1][0] and sbp1[0][2] == sbp2[0][2] and sbp1[1][2] == sbp2[1][2]:
            # if left and right breakpoint locations are within d
            if (abs(sbp1[0][1] - sbp2[0][1]) + abs(sbp1[1][1] - sbp2[1][1]) < d) or (abs(sbp1[0][1] - sbp2[1][1]) + abs(sbp1[1][1] - sbp2[0][1]) < d):
                return True
        return False

class Cycle(object):
    
    def __init__(self,cn,length,breakpoints,circular,trivial):
        self.cn=float(cn)
        self.length=int(length)
        self.breakpoints=breakpoints # list of breakpoints
        self.circular=bool(circular)
        self.trivial=bool(trivial)
        
    def __str__(self):
        bp_expanded=','.join(str(b) for b in self.breakpoints)
        return(f'Copy_count={self.cn};Length={self.length}bp;Breakpoints={bp_expanded};Circular={self.circular};Trivial={self.trivial}')
        
    def __add__(self,o):
        return Cycle(cn=np.mean([self.cn,o.cn]), length=self.length+o.length, breakpoints=self.breakpoints+o.breakpoints,
                    circular = self.circular&o.circular, trivial = self.trivial&o.trivial)
    
    def diff(self,o,d=1000):
        df = np.zeros((len(self.breakpoints),len(o.breakpoints)))
        for i in range(len(self.breakpoints)):
            for j in range(len(o.breakpoints)):
                df[i,j]=self.breakpoints[i].d_similar(o.breakpoints[j],d)
        return df
    
    def plot_diff(self,other,d=1000):
        df = self.diff(other,d=d)
        #plt.imshow(binary_matrix, cmap='viridis', interpolation='nearest')
        plt.imshow(df)
        # Set x-axis labels
        plt.yticks(np.arange(len(self.breakpoints)), map(str,self.breakpoints))

        # Set y-axis labels
        plt.xticks(np.arange(len(other.breakpoints)), map(str,other.breakpoints), rotation=90)

        # Add labels to the axes
        plt.ylabel("Self")
        plt.xlabel("Other")
        
        plt.grid(True, which='both', linestyle='-', linewidth=1, color='white')

        # Show the plot
        plt.show()
        
class Converted_Cycles(object):
    def __init__(self,cycles,segments,intervals):
        self.cycles=cycles
        self.segments=segments
        self.intervals=intervals
    
    def __init__(self,path):
        '''
        path or iobuffer to a _BPG_converted_cycles.txt file
        '''
        self.intervals=[('Source',0,0)] #(chr start end)
        self.segments=[('Source',0,0)] #(chr start end)
        self.cycles=[] # Cycles
        with open(path,'r') as file:
            for line in file:
                if line.startswith('Interval'):
                    self.parse_interval(line)
                elif line.startswith('Segment'):
                    self.parse_segment(line)
                elif line.startswith('Cycle'):
                    self.parse_cycle(line)
                else:
                    continue
                        
    def parse_segment(self,line):
        line=line.strip().split()
        self.segments.append((line[2],int(line[3]),int(line[4])))
    def parse_interval(self,line):
        line=line.strip().split()
        self.intervals.append((line[2],int(line[3]),int(line[4])))
    def parse_breakpoints(self,subline):
        subline=subline.split(',')
        breakpoints=[]
        for i in range(len(subline)):
            c=(int(subline[i][:-1]),subline[i][-1]) #current
            n_i = i+1 if i+1 < len(subline) else 0
            n=(int(subline[n_i][:-1]),subline[n_i][-1]) #next
            if c[1]=='+' and n[1]=='+' and n[0]-c[0]==1:
                continue
            elif c[1]=='-' and n[1]=='-' and c[0]-n[0]==1:
                continue
            else:
                breakpoints.append(Breakpoint(
                    self.segments[c[0]][0],self.segments[c[0]][2] if c[1]=='+' else self.segments[c[0]][1],c[1]=='+',
                    self.segments[n[0]][0],self.segments[n[0]][1] if n[1]=='+' else self.segments[n[0]][2],n[1]!='+',
                    None
                ))
        return breakpoints
        
    def parse_cycle(self,line):
        line=list(map(lambda x: x.split('='), line.strip().split(';')))
        self.cycles.append(Cycle(cn=line[1][1],
                                length=line[2][1][:-2],
                                breakpoints=self.parse_breakpoints(line[3][1]),
                                circular=line[4][1] == 'TRUE',
                                trivial=line[5][1] == 'TRUE'))
        
        
BS_W37QBA12_parse = Converted_Cycles('../CycleViz/BS_W37QBA12/BS_W37QBA12_amplicon1_BPG_converted_cycles.txt')
BS_2J4FG4HV_parse = Converted_Cycles('../CycleViz/BS_2J4FG4HV/BS_2J4FG4HV_amplicon1_BPG_converted_cycles.txt')
BS_5JC116NM_parse = Converted_Cycles('../CycleViz/BS_5JC116NM/BS_5JC116NM_amplicon1_BPG_converted_cycles.txt')

empty=Cycle(np.nan,0,[],True,True)
BS_W37QBA12_ecdna = sum(BS_W37QBA12_parse.cycles[:3],empty)
BS_2J4FG4HV_ecdna = sum(BS_2J4FG4HV_parse.cycles[:3],empty)
BS_5JC116NM_ecdna = sum(BS_5JC116NM_parse.cycles[:6],empty)


In [ ]:
SJRHB012_D_parse = Converted_Cycles('../CycleViz/SJRHB012_D/SJRHB012_D_amplicon1_BPG_converted_cycles.txt')
SJRHB012_R_parse1 = Converted_Cycles('../CycleViz/SJRHB012_S/SJRHB012_S_amplicon1_BPG_converted_cycles.txt')
SJRHB012_R_parse2 = Converted_Cycles('../CycleViz/SJRHB012_S/SJRHB012_S_amplicon2_BPG_converted_cycles.txt')

In [ ]:
SJRHB012_D_ecdna1 = SJRHB012_D_parse.cycles[3]
SJRHB012_R_ecdna1 = SJRHB012_R_parse1.cycles[0]
SJRHB012_D_ecdna2 = SJRHB012_D_parse.cycles[0]
SJRHB012_R_ecdna2 = SJRHB012_R_parse2.cycles[0]

In [ ]:
SJRHB012_D_ecdna1.plot_diff(SJRHB012_R_ecdna1)

In [ ]:
SJRHB012_D_ecdna2.plot_diff(SJRHB012_R_ecdna2)

In [ ]:
# primary vs primary
BS_2J4FG4HV_ecdna.plot_diff(BS_W37QBA12_ecdna)

In [ ]:
# primary vs relapse
BS_2J4FG4HV_ecdna.plot_diff(BS_5JC116NM_ecdna)

# Dead code

In [ ]:
# how many diagnosis tumors have ecDNA? 126 / 1291
# Update 11/2024: 209 / 2559
# No filter on included tumor types
def primaries_w_ecDNA(tumor_types=None,verbose=True):
    if tumor_types is None:
        df1 = BIOSAMPLES[BIOSAMPLES.tumor_history == "Diagnosis"].groupby('patient_id').agg(aggregated_value=('amplicon_class', lambda x: (x == 'ecDNA').sum())).reset_index()
    else:
        df1 = BIOSAMPLES[(BIOSAMPLES.tumor_history == "Diagnosis") & (BIOSAMPLES.cancer_type.isin(tumor_types))].groupby('patient_id').agg(aggregated_value=('amplicon_class', lambda x: (x == 'ecDNA').sum())).reset_index()
    df1["primary_ecDNA"] = df1.aggregated_value > 0
    df1.set_index("patient_id",inplace=True)
    df1.drop("aggregated_value",axis=1,inplace=True)
    a = len(df1[df1.primary_ecDNA])
    b = len(df1)
    if verbose:
        print(f"{a} of {b} ({round(a/b*100,1)}%) primary tumors have ecDNA")
    return(df1)
p = primaries_w_ecDNA()

In [ ]:
# how many secondary tumors have ecDNA? 33 / 199
# Update 11/2024: 57 / 511
def secondaries_w_ecDNA(verbose=True):
    df2 = BIOSAMPLES[BIOSAMPLES.tumor_history.isin(["Recurrence","Progressive","Relapse","Metastasis"])].groupby('patient_id').agg(aggregated_value=('amplicon_class', lambda x: (x == 'ecDNA').sum())).reset_index()
    df2["secondary_ecDNA"] = df2.aggregated_value > 0
    df2.set_index("patient_id",inplace=True)
    df2.drop("aggregated_value",axis=1,inplace=True)
    a = len(df2[df2.secondary_ecDNA])
    b = len(df2)
    if verbose:
        print(f"{a} of {b} ({round(a/b*100,1)}%) secondary tumors have ecDNA")
    return(df2)
s = secondaries_w_ecDNA()

In [ ]:
# longitudinal samples
def get_longitudinal_primary_secondary_pairs(verbose=True):
    df1 = primaries_w_ecDNA(verbose=False)
    df2 = secondaries_w_ecDNA(False)
    df=df1.merge(df2,how='inner',left_index=True,right_index=True)
    if verbose:
        print(f'{len(df)} primary/secondary pairs')
    return df
get_longitudinal_primary_secondary_pairs()

In [ ]:
def plot_primary_secondary_amp_status(df,ylabel=None):
    '''
    Barplot illustrating primary/secondary amplicon rates without coloring by tumor type.
    '''
    frac = df.groupby("history")["amp"].mean()
    
    fig, ax = plt.subplots()
    ax.bar(frac.index, frac.values,width=0.9)
    
    ax.set_xlabel(None)
    ax.set_ylabel(f"Fraction with {ylabel}")
    ax.set_ylim(0, 0.2)
    
    # remove top and right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    plt.show()
plot_primary_secondary_amp_status(df,"ecDNA")

In [ ]:
df.head()